This code needs to run with a custom environment as analysis3-unstable does not have NEMOSIS.

In [1]:
import os
import glob
import pandas as pd
from datetime import datetime, timedelta
from nemosis import dynamic_data_compiler, static_table, cache_compiler
from concurrent.futures import ThreadPoolExecutor, as_completed


# === CONFIGURATION ===
write_path = '/scratch/ng72/ms5578/time_series'
raw_NEM_cache = '/scratch/ng72/ms5578/nemosis_cache'

In [2]:
gen_list = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/gen_details.csv')

In [3]:
start_date = datetime(2016, 7, 1)
end_date = datetime(2019, 6, 30)

# === Output directory setup ===
output_dir = os.path.join(write_path, "nem_generation")
os.makedirs(output_dir, exist_ok=True)
for f in glob.glob(f"{output_dir}/*.csv"):
    os.remove(f)

# === Time chunk generator ===
def month_range(start, end):
    current = start
    while current < end:
        next_month = (current.replace(day=1) + timedelta(days=32)).replace(day=1)
        yield current, min(next_month - timedelta(seconds=1), end)
        current = next_month

# === Worker function ===
def process_chunk(gen_list, chunk_start, chunk_end):
    try:
        gen_df = dynamic_data_compiler(
            start_time=chunk_start.strftime("%Y/%m/%d %H:%M:%S"),
            end_time=chunk_end.strftime("%Y/%m/%d %H:%M:%S"),
            table_name='DISPATCHLOAD',
            raw_data_location=raw_NEM_cache,
            select_columns=['SETTLEMENTDATE', 'DUID', 'INITIALMW', 'TOTALCLEARED', 'AGCSTATUS'],
            format='feather'
        )
    except Exception as e:
        print(f"❌ Failed: {chunk_start} – {chunk_end}: {e}")
        return

    if gen_df.empty:
        print(f"⚠️ Empty: {chunk_start} – {chunk_end}")
        return

    gen_df = gen_df.rename(columns={'SETTLEMENTDATE': 'time'})
    gen_df['time'] = pd.to_datetime(gen_df['time'])
    gen_df['TOTALMWh'] = gen_df['INITIALMW'] * 5 / 60
    gen_df['TOTALCLEARED'] = gen_df['TOTALCLEARED'] * 5 / 60
    gen_df = gen_df.drop(columns=['INITIALMW'])
    
    agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'min'}
    grouped = gen_df.groupby(['DUID', pd.Grouper(key='time', freq='1h')]).agg(agg_func).reset_index()

    for duid, df_duid in grouped.groupby("DUID"):
        safe_duid = duid.replace("/", "_").replace("\\", "_").replace('#','_')
        out_path = os.path.join(output_dir, f"{safe_duid}.csv")

        write_header = not os.path.isfile(out_path) or os.path.getsize(out_path) == 0
        df_duid.to_csv(out_path, mode='a', header=write_header, index=False)

    print(f"✅ Processed chunk: {chunk_start} – {chunk_end}")


# === Run threads ===
time_chunks = list(month_range(start_date, end_date))
        
for s, e in time_chunks:
    process_chunk(gen_list, s, e)

INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2016-07-01 00:00:00 – 2016-07-31 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2016-08-01 00:00:00 – 2016-08-31 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2016-09-01 00:00:00 – 2016-09-30 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2016-10-01 00:00:00 – 2016-10-31 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2016-11-01 00:00:00 – 2016-11-30 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2016-12-01 00:00:00 – 2016-12-31 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Returning DISPATCHLOAD.
✅ Processed chunk: 2017-01-01 00:00:00 – 2017-01-31 23:59:59
INFO: Compiling data for table DISPATCHLOAD
INFO: Retur

In [4]:
gen_df = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/gen_details.csv')
gen_df.drop(gen_df.filter(regex="Unname"),axis=1, inplace=True)
gen_df.columns = (
    gen_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
    )

gen_df = gen_df.rename(columns={'duid': 'DUID'})
gen_df.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_details.csv',
                index=False)

In [5]:
sdate, edate = "2016/07/01 00:00:00", "2019/06/30 23:59:59"

dem_df = dynamic_data_compiler(start_time=sdate,
                                   end_time=edate,
                                   table_name='DISPATCHREGIONSUM',
                                   raw_data_location=raw_NEM_cache,
                                   select_columns=['REGIONID', 'SETTLEMENTDATE', 'TOTALDEMAND'],
                                   format='feather')


dem_df = dem_df.rename(columns={'SETTLEMENTDATE': "time"})
dem_df['time'] = pd.to_datetime(dem_df['time'])
dem_df["REGIONID"] = dem_df["REGIONID"].astype("category")

dem_df['TOTALDEMAND'] = dem_df['TOTALDEMAND'].apply(lambda x: x*5/60)

dem_df = dem_df.set_index("time").groupby("REGIONID").resample("1h").sum()

dem_df = dem_df.reset_index()

dem_df.to_csv(f'{write_path}/state_demand.csv',
             index=False)

INFO: Compiling data for table DISPATCHREGIONSUM
INFO: Returning DISPATCHREGIONSUM.


/jobfs/145004933.gadi-pbs/ipykernel_3718213/699573750.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dem_df = dem_df.set_index("time").groupby("REGIONID").resample("1h").sum()
